In [77]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Project root on path so the lora_playground package imports work from notebooks/.
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# All shared loaders + plotters live in lora_playground.plot_utils.
# Each notebook section just supplies (cfg, evs) runs + a group_key callable.
from lora_playground.plot_utils import (
    DIVERGE_THRESHOLD,
    has_runs,
    load_sweep,
    max_loss,
    merge_runs,
    parse_flag,
    plot_best_eta_curves,
    plot_eta_vs_final,
    split_diverged,
    two_panel_sweep_figure,
)

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

In [ ]:
# Color palette for the all-optimizer comparison. Each section picks the
# subset relevant to it.
OPTIM_COLORS = {
    "adamw":            "#1f77b4",
    "scaled-lora":      "#ff7f0e",
    "lin-lora":         "#2ca02c",
    "adam-scaled-lora": "#d62728",
    "adam-lin-lora":    "#9467bd",
    "muon-lora":        "#e377c2",
    "diag-scaled-lora": "#17becf",
    "kron-grad-lora":   "#bcbd22",
    "psi-lora":         "#8c564b",
    "galore-adamw":     "#7f7f7f",
}

# m → color for the LoRA+ co-sweep
M_COLORS = {1: plt.cm.plasma(0.15), 4: plt.cm.plasma(0.50), 16: plt.cm.plasma(0.85)}

# SVD oracle modes
MODE_COLORS = {"svd_step_oracle": "#ff7f0e", "svd_cumulative_oracle": "#2ca02c"}

## LoRA+ (η, m) co-sweep — AdamW, r=16, 1 epoch

In [ ]:
# Merge AdamW LoRA+ runs across all sweeps that varied (η, m). Filter to
# optimizer="adamw" only. Dedup by (m, η). Earlier groups override later.
LP_SOURCES = (
    "loraplus_lowlr_2k",        # m ∈ {4, 16} × η ∈ {1e-5, 3e-5} (extension)
    "loraplus_2k_1ep",          # m ∈ {1, 4, 16} × η ∈ {1e-4, 3e-4} (original)
    "lr_sweep_2k",              # m=1 × η ∈ {3e-5..1e-3}
    "optim_compare_high_eta_2k",# m=1 × η ∈ {3e-3, 1e-2}
    "new_optimizers_2k",        # m=1 × η ∈ {3e-5..1e-3}
)

lp_runs = merge_runs(
    LP_SOURCES,
    key_fn=lambda c: (float(c.get("lora_plus_multiplier", 1.0)), float(c["lr"])),
    filter_fn=lambda c: c.get("optimizer") == "adamw",
)
print(f"Loaded {len(lp_runs)} AdamW LoRA+ runs (merged)")
for cfg, evs in sorted(lp_runs, key=lambda x: (x[0].get("lora_plus_multiplier", 1), x[0]["lr"])):
    print(f"  m={cfg.get('lora_plus_multiplier', 1):.0f}  η={cfg['lr']:.0e}  "
          f"final={evs[-1]['eval_loss']:.4f}")

In [ ]:
def _lp_group(cfg):
    return f"m={int(round(cfg.get('lora_plus_multiplier', 1.0)))}"

# Build a color map keyed by "m={k}" labels from the M_COLORS palette.
LP_COLOR_MAP = {f"m={m}": c for m, c in M_COLORS.items()}

fig, axes, n_keep, n_drop = two_panel_sweep_figure(
    lp_runs,
    group_key_fn=_lp_group,
    color_map=LP_COLOR_MAP,
    suptitle="LoRA+ (η, m) — AdamW, r=16, 2000 steps",
    left_title="Final eval loss vs η, per m",
    right_title="Best η per m — training curves",
    label_fn=lambda c: f"{_lp_group(c)} η={c['lr']:.0e}",
)
plt.show()

## All optimizers — η sweep, r=16, 2000 steps

η ∈ {3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2}, seed=0. All optimizers compared on the
same axis. Sources merged across `lr_sweep_2k`, `optim_compare_high_eta_2k`,
`muon_lowlr_2k`, `new_optimizers_high_eta_2k`, `new_optimizers_2k`, `psi_lora_2k`,
`galore_fixed_2k`. Diverged runs (max eval > 1.5) filtered.

**Honest naming.** What we previously called `psi-lora` was just diagonal D_V/D_U scaling —
renamed to **`diag-scaled-lora`**. `kfac-lora` was a custom variant (r×r gradient outer
products) — renamed to **`kron-grad-lora`**. The `psi-lora` slot now points to the paper
Algorithm 3 port (F-LoRSUM proximal subspace iteration with K-FAC metrics + low-rank
momentum, eq. 14 of arXiv 2602.16456).

**Muon-LoRA NS bug fix.** Original `_newton_schulz` re-multiplied output by input Frobenius
norm, breaking scale-invariance — combined with PEFT's B=0 init the updates collapsed and
η was nearly irrelevant. Fixed: NS output now has Frobenius norm √r regardless of input.

**GaLore** (`galore-adamw`) uses a different training mode — full dense weights with rank-r
gradient projection refreshed every 200 steps. Included for direct comparison; it's ~3× slower
per step than LoRA-mode optimizers due to backprop through full weights.

In [ ]:
# Sources, in priority order for (optimizer, η) deduplication. Earlier groups
# override later ones. Comment lines describe each group's contribution.
SOURCE_GROUPS = (
    "psi_lora_2k",                # PSI-LoRA Algorithm 3 port
    "muon_lowlr_2k",              # bug-fixed muon at low η
    "boundary_extend3_2k",        # lin/scaled at η ∈ {10, 30}
    "boundary_extend2_2k",        # diag/lin/scaled at η ∈ {1, 3}
    "boundary_extend_2k",         # diag/lin/scaled at η ∈ {3e-2, 1e-1, 3e-1}
    "new_optimizers_high_eta_2k", # diag/kron/muon at η ∈ {3e-3, 1e-2, 3e-2}
    "galore_fixed_2k",            # GaLore matched to ~/GaLore (post-fix)
    "optim_compare_high_eta_2k",  # original 5 optimizers at η ∈ {3e-3, 1e-2}
    "lr_sweep_2k",                # original 5 optimizers at low η
    "new_optimizers_2k",          # legacy psi/kfac at low η (renamed via alias)
    "galore_2k",                  # buggy GaLore (superseded by galore_fixed)
)

# Historical logs labeled "psi-lora"/"kfac-lora" were the diagonal-scaled /
# Kron-grad ablations, NOT the paper algorithms. Alias them to honest names
# so the legend reflects what was computed. The "psi-lora" slot is reserved
# for runs from psi_lora_2k (the actual Algorithm 3 port).
RENAME_ALIAS = {"psi-lora": "diag-scaled-lora", "kfac-lora": "kron-grad-lora"}
NEW_PSI_GROUPS = {"psi_lora_2k"}


def _alias_optimizer(cfg, group):
    if group not in NEW_PSI_GROUPS:
        cfg["optimizer"] = RENAME_ALIAS.get(cfg["optimizer"], cfg["optimizer"])

all_runs = merge_runs(
    SOURCE_GROUPS,
    key_fn=lambda c: (c["optimizer"], float(c["lr"])),
    filter_fn=lambda c: c["optimizer"] in OPTIM_COLORS,
    cfg_postprocess=_alias_optimizer,
)

print(f"Loaded {len(all_runs)} runs across {len({c['optimizer'] for c, _ in all_runs})} optimizers")
for cfg, evs in sorted(all_runs, key=lambda x: (x[0]["optimizer"], x[0]["lr"])):
    flag = " DIVERGED" if max_loss(evs) >= DIVERGE_THRESHOLD else ""
    print(f"  {cfg['optimizer']:<18s}  η={cfg['lr']:.0e}  step={evs[-1]['step']}  "
          f"final={evs[-1]['eval_loss']:.4f}{flag}")

In [ ]:
fig, axes, n_keep, n_drop = two_panel_sweep_figure(
    all_runs,
    group_key_fn=lambda c: c["optimizer"],
    color_map=OPTIM_COLORS,
    suptitle="All optimizers — η sweep, r=16, 2000 steps",
    label_fn=lambda c: f"{c['optimizer']:<18s} η={c['lr']:.0e}",
)
plt.show()

## SVD oracle — step vs cumulative, η sweep, r=16, 1 epoch

`svd_step_oracle`: each AdamW step projected to rank r via randomized SVD (niter=4).  
`svd_cumulative_oracle`: cumulative displacement from init projected to rank r.  
Both use full dense weights (not LoRA adapters). Reference: best AdamW LoRA at η=3e-4.

In [ ]:
svd_runs = load_sweep("svd_sweep_2k_1ep")
print(f"Loaded {len(svd_runs)} SVD oracle runs")
for cfg, evs in sorted(svd_runs, key=lambda x: (x[0]["training_mode"], x[0]["lr"])):
    flag = " DIVERGED" if max_loss(evs) >= DIVERGE_THRESHOLD else ""
    print(f"  {cfg['training_mode']:<25s}  η={cfg['lr']:.0e}  "
          f"final={evs[-1]['eval_loss']:.4f}{flag}")

In [ ]:
# Use the AdamW LoRA at η=3e-4 from the all-optimizer sweep as a reference
# horizontal/curve overlay.
adamw_ref = next(((c, e) for c, e in all_runs
                  if c["optimizer"] == "adamw" and abs(c["lr"] - 3e-4) < 1e-9), None)

hlines = []
ref_curves = []
if adamw_ref is not None:
    cfg, evs = adamw_ref
    hlines.append(("AdamW LoRA η=3e-4 (ref)", evs[-1]["eval_loss"], "black"))
    ref_curves.append(("AdamW LoRA η=3e-4", evs, "black", ":"))

fig, axes, n_keep, n_drop = two_panel_sweep_figure(
    svd_runs,
    group_key_fn=lambda c: c["training_mode"],
    color_map=MODE_COLORS,
    suptitle="SVD oracle — step vs cumulative, η sweep, r=16, 1 epoch",
    hlines=hlines,
    ref_curves=ref_curves,
    label_fn=lambda c: f"{c['training_mode']:<25s} η={c['lr']:.0e}",
)
plt.show()

## Muon-LoRA variants — beat AdamW campaign

Hypothesis-driven sweep:
- **Tier 1 (`muon_loraplus_2k`)**: H1 — LoRA+ asymmetry on B's lr (m ∈ {4, 16}). Baseline m=1 reused from earlier groups.
- **Tier 2 (`muon_nsoff_2k`)**: H3 sanity — `--muon_ns_steps 0` ⇒ momentum SGD (NS disabled). If matches Muon-LoRA, the orthogonalization is irrelevant.
- **Tier 3 (`product_muon_2k`)**: H2 — `ProductMuonLoRA`, NS on the merged-direction proxy `D = (1/scale)·m_B·(AAᵀ+δI)⁻¹·A` then Sylvester-recovered. Theory: `docs/theory/main.tex` line 622+.
- **Tier 4 (`adam_muon_2k`)**: H4 — `AdamMuonLoRA`, NS on Adam's m̂/(√v̂+ε) direction.

Reference frontiers (from earlier sweeps): AdamW @ 0.7579, adam-lin-lora @ 0.7564. Goal: **strictly beat** both.

In [ ]:
def _stamp_muon_meta(cfg, group):
    """Inject muon_ns_steps from --muon_ns_steps (default 5)."""
    v = parse_flag(cfg.get("command", ""), "--muon_ns_steps")
    cfg["muon_ns_steps"] = int(v) if v is not None else 5

MUON_SOURCES = (
    "muon_loraplus_lowlr_2k",      # m ∈ {4, 16} × η ∈ {1e-4, 3e-4} (extension)
    "muon_loraplus_2k",            # m ∈ {4, 16} × η ∈ {1e-3, 3e-3, 1e-2}
    "muon_nsoff_2k",               # ns_steps=0 sanity check
    "product_muon_2k",             # H2: ProductMuonLoRA
    "adam_muon_2k",                # H4: AdamMuonLoRA
    "muon_lowlr_2k",               # m=1 baseline at η ∈ {3e-5..1e-3}
    "new_optimizers_high_eta_2k",  # m=1 baseline at η ∈ {3e-3..3e-2}
)

muon_runs = merge_runs(
    MUON_SOURCES,
    key_fn=lambda c: (c["optimizer"], float(c["lr"]),
                      float(c.get("lora_plus_multiplier", 1.0)), c["muon_ns_steps"]),
    filter_fn=lambda c: c["optimizer"] in {"muon-lora", "product-muon-lora", "adam-muon-lora"},
    cfg_postprocess=_stamp_muon_meta,
)

print(f"Loaded {len(muon_runs)} Muon-family runs")
for cfg, evs in sorted(muon_runs, key=lambda x: (x[0]["optimizer"], x[0]["muon_ns_steps"],
                                                  x[0]["lora_plus_multiplier"], x[0]["lr"])):
    flag = " DIVERGED" if max_loss(evs) >= DIVERGE_THRESHOLD else ""
    print(f"  {cfg['optimizer']:<18s}  η={cfg['lr']:.0e}  "
          f"m={float(cfg['lora_plus_multiplier']):>4.1f}  "
          f"ns={cfg['muon_ns_steps']}  final={evs[-1]['eval_loss']:.4f}{flag}")

In [ ]:
def _muon_variant(cfg):
    opt = cfg["optimizer"]
    m = float(cfg.get("lora_plus_multiplier", 1.0))
    ns = cfg["muon_ns_steps"]
    base = {"muon-lora": "muon", "product-muon-lora": "product-muon",
            "adam-muon-lora": "adam-muon"}[opt]
    parts = [base]
    if m != 1.0:
        parts.append(f"m={int(m)}")
    if ns == 0:
        parts.append("ns=0")
    return "+".join(parts)


VARIANT_COLORS = {
    "muon":              "#e377c2",
    "muon+m=4":          "#d62728",
    "muon+m=16":         "#8c2d04",
    "muon+ns=0":         "#bababa",
    "product-muon":      "#1f77b4",
    "product-muon+m=4":  "#0d3d66",
    "adam-muon":         "#2ca02c",
    "adam-muon+m=4":     "#0d4f0d",
}

# Reference frontiers from the all-optimizer sweep.
adamw_ref = next(((c, e) for c, e in all_runs if c["optimizer"] == "adamw"), None)
adamlin_ref = next(((c, e) for c, e in all_runs if c["optimizer"] == "adam-lin-lora"), None)

# Pick best (lowest final) reference across η for each.
def _best(runs_iter):
    best = None
    for c, e in runs_iter:
        if best is None or e[-1]["eval_loss"] < best[1][-1]["eval_loss"]:
            best = (c, e)
    return best

adamw_best = _best([(c, e) for c, e in all_runs if c["optimizer"] == "adamw"])
adamlin_best = _best([(c, e) for c, e in all_runs if c["optimizer"] == "adam-lin-lora"])

hlines, ref_curves = [], []
if adamw_best is not None:
    hlines.append((f"AdamW best ({adamw_best[1][-1]['eval_loss']:.4f})",
                   adamw_best[1][-1]["eval_loss"], "black"))
    ref_curves.append(("AdamW best", adamw_best[1], "black", ":"))
if adamlin_best is not None:
    hlines.append((f"adam-lin-lora best ({adamlin_best[1][-1]['eval_loss']:.4f})",
                   adamlin_best[1][-1]["eval_loss"], "purple"))
    ref_curves.append(("adam-lin-lora best", adamlin_best[1], "purple", ":"))

fig, axes, n_keep, n_drop = two_panel_sweep_figure(
    muon_runs,
    group_key_fn=_muon_variant,
    color_map=VARIANT_COLORS,
    suptitle="Muon-LoRA campaign — beat-AdamW",
    hlines=hlines,
    ref_curves=ref_curves,
    label_fn=lambda c: f"{_muon_variant(c):<22s} η={c['lr']:.0e}",
)
plt.show()

# Verdict table.
keep, _ = split_diverged(muon_runs)
adamw_floor = adamw_best[1][-1]["eval_loss"] if adamw_best else float("inf")
adamlin_floor = adamlin_best[1][-1]["eval_loss"] if adamlin_best else float("inf")
best_per_v = {}
for cfg, evs in keep:
    v = _muon_variant(cfg)
    fl = evs[-1]["eval_loss"]
    if v not in best_per_v or fl < best_per_v[v][2]:
        best_per_v[v] = (cfg, evs, fl)
print("\n=== Did we beat AdamW? ===")
for v, (cfg, evs, fl) in sorted(best_per_v.items(), key=lambda kv: kv[1][2]):
    beat_adamw = "✅" if fl < adamw_floor else "❌"
    beat_adamlin = "✅" if fl < adamlin_floor else "❌"
    print(f"  {v:<22s}  best={fl:.4f}  vs AdamW({adamw_floor:.4f}): {beat_adamw}  "
          f"vs adam-lin({adamlin_floor:.4f}): {beat_adamlin}")

## H4 *-Post + H5 matrix-Adam + Polar-Product investigation

Cross-investigation leaderboard for the lin/scaled-lora extensions:
- **`*-post`**: Adam(m,v) on raw ∇, then geometric solve (Sylvester / Gram) on Adam's step. RMS-aligned step magnitude.
- **`*-matrix`**: per-pair scalar v̂ instead of per-coord — preserves direction by sacrificing Adam's per-coord adaptation.
- **`adam-polar-product-lora`**: theory's closed-form spectral-product update (lemma at `docs/theory/main.tex` line 622-660). Polar (NS) sandwiched between two factors of S^{-1/2}, fed Adam's denoised direction.

See `docs/notes/optimizer_synthesis.md` for the cross-investigation synthesis.

In [ ]:
# Load all post-Adam and polar-product sweep groups
ext_groups = [
    "h4_post_2k", "h4_post_rmsalign_2k",
    "h5_matrix_2k", "h5_matrix_r64_2k",
    "polar_product_2k",
    "h3_rsweep_2k",                           # for r-scan baselines
    "optim_compare_high_eta_2k", "lr_sweep_2k",  # for r=16 baselines
]
ext_runs = []
for grp in ext_groups:
    if not _has_runs(grp):
        continue
    for cfg, evs in load_sweep(grp):
        ev = evs[-1] if evs else None
        if ev is None:
            continue
        cfg["_lora_r"] = int(cfg.get("lora_r", 16))
        cfg["_step"]   = ev["step"]
        cfg["_final"]  = ev["eval_loss"]
        cfg["_group"]  = grp
        ext_runs.append((cfg, evs))

print(f"Loaded {len(ext_runs)} runs across {len(ext_groups)} groups")
# Best-η per (optimizer, r) at step 2000
best = {}
for cfg, evs in ext_runs:
    if cfg["_step"] < 2000:
        continue
    k = (cfg["optimizer"], cfg["_lora_r"])
    if k not in best or cfg["_final"] < best[k][2]:
        best[k] = (cfg, evs, cfg["_final"])

print()
print(f"{'rank':>4}  {'optimizer':<28s}  {'r':>3}  {'best η':>9s}  {'final':>7s}  vs AdamW r=16")
ADAMW_r16 = 0.7579
for i, ((opt, r), (cfg, evs, fl)) in enumerate(sorted(best.items(), key=lambda kv: kv[1][2]), 1):
    delta = fl - ADAMW_r16
    flag = "✅" if delta < -0.005 else ("≈" if abs(delta) < 0.005 else "❌")
    print(f"{i:>4d}  {opt:<28s}  {r:>3d}  {cfg['lr']:>9.0e}  {fl:>7.4f}  Δ={delta:+.4f} {flag}")

In [ ]:
# Trajectories at best-η, faceted by rank.
# Each panel uses ITS OWN per-rank AdamW reference — comparing r=64 results
# against the r=16 AdamW frontier is not apples-to-apples.
RANKS_TO_PLOT = [16, 64]
COLOR_BY_FAMILY = {
    "adamw":                       "#1f77b4",
    "adam-lin-lora":               "#9467bd",
    "adam-scaled-lora":            "#d62728",
    "adam-lin-lora-post":          "#2ca02c",
    "adam-scaled-lora-post":       "#ff7f0e",
    "adam-lin-lora-matrix":        "#17becf",
    "adam-scaled-lora-matrix":     "#bcbd22",
    "adam-polar-product-lora":     "#8c564b",
    "polar-product-lora":          "#e377c2",
}

fig, axes = plt.subplots(1, len(RANKS_TO_PLOT),
                          figsize=(7 * len(RANKS_TO_PLOT), 5), sharey=True)
for ax, r_target in zip(axes, RANKS_TO_PLOT):
    # Per-rank AdamW reference. Falls back to None if no AdamW run at that r.
    adamw_ref = best.get(("adamw", r_target))
    for opt in COLOR_BY_FAMILY:
        entry = best.get((opt, r_target))
        if entry is None:
            continue
        cfg, evs, fl = entry
        steps = [e["step"] for e in evs]
        losses = [e["eval_loss"] for e in evs]
        ax.plot(steps, losses, color=COLOR_BY_FAMILY[opt], lw=1.8,
                label=f"{opt} η={cfg['lr']:.0e}: {fl:.4f}", marker="o", markersize=2)
    if adamw_ref is not None:
        adamw_floor = adamw_ref[2]
        ax.axhline(adamw_floor, ls=":", color="black", lw=1,
                   label=f"AdamW r={r_target} = {adamw_floor:.4f}")
        # Strict-win bar = 1% below the per-rank AdamW floor.
        ax.axhline(adamw_floor - 0.01, ls="--", color="grey", lw=0.8,
                   label=f"strict-win bar = {adamw_floor - 0.01:.4f}")
    ax.set_xlabel("step")
    if r_target == RANKS_TO_PLOT[0]:
        ax.set_ylabel("eval loss")
    ax.set_title(f"r = {r_target}")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5),
              fontsize=7, frameon=True)

fig.suptitle("Best-η trajectories per optimizer family — per-rank AdamW reference")
fig.tight_layout()
plt.show()